In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import mne
import xarray as xr
import seaborn as sns
from pathlib import Path
from matplotlib.ticker import FuncFormatter, MaxNLocator
from deepjr.simulation import jr_typical_param, JRSimulator, EventRelatedExp
import matplotlib.pyplot as plt
import xarray as xr
import pandas as pd
import numpy as np
import seaborn as sns
from matplotlib.ticker import FuncFormatter
from deepjr.jr_inv_lstm_model import JRInvDataLoader

# Set parameters
path = "./deepjr_training_data"
estim_params = ('A_e', 'A_i', 'b_e', 'b_i', 'a_1', 'a_2', 'a_3', 'a_4')
nb_sims = 200
output_dir = "./jr_transformer_output"
noise_fact = 0      


# ============================================================
# 1. SETUP: DEFAULT PARAMETERS & SIMULATION SETTINGS
# ============================================================
default_params = dict(jr_typical_param)
channel_of_interest = "O2"
fontsize = 22


# LaTeX labels for display (subscripts)
latex_labels = {
    'A_e': r'$A_{e}$',
    'A_i': r'$A_{i}$',
    'b_e': r'$b_{e}$',
    'b_i': r'$b_{i}$',
    'a_1': r'$a_{1}$',
    'a_2': r'$a_{2}$',
    'a_3': r'$a_{3}$',
    'a_4': r'$a_{4}$'
}

jr_param_ranges = {
    'A_e': (2.6 * 1e-3, 9.75 * 1e-3),  # Convert from mV to V
    'A_i': (17.6 * 1e-3, 110.0 * 1e-3), # Convert from mV to V
    'b_e': (5, 150),         # s^-1
    'b_i': (25, 75),         # s^-1
    'C': (65, 1350),         # dimensionless
    'a_1': (0.5, 1.5),       # dimensionless
    'a_2': (0.4, 1.2),       # dimensionless
    'a_3': (0.125, 0.375),   # dimensionless
    'a_4': (0.125, 0.375),}

# ============================================================
# 2. FUNCTION: RUN SENSITIVITY FOR ONE PARAMETER
# ============================================================
def run_sensitivity_for_param(param, default_params, nb_sims, noise_fact, jr_param_ranges):
    """
    For the given parameter, vary its value across its specific range from Table 1,
    while keeping all other parameters at their default values.
    
    Args:
        param: Parameter name to vary
        default_params: Dictionary of default parameter values
        nb_sims: Number of simulations to run
        noise_fact: Noise factor
        jr_param_ranges: Dictionary mapping parameter names to (min, max) tuples
    
    Returns:
        - evoked_list: a list of MNE Evoked objects (one per simulation)
        - param_vals: a list of the corresponding parameter values
    """
    #print(default_params)
    
    # Use parameter-specific range from Table 1 instead of fixed 50-150% range
    min_val, max_val = jr_param_ranges[param]
    sim_values = np.linspace(min_val, max_val, nb_sims)
    
    evoked_list = []
    param_vals = []
    print(f"Running sensitivity analysis for {param} across range [{min_val}, {max_val}] ...")
    
    for val in sim_values:
        new_params = default_params.copy()
        new_params[param] = val
        print(new_params)
        # Reinitialize the simulator for a fresh state.
        jr_sim = JRSimulator()
        er_exp = EventRelatedExp(jr_sim.info)
        jr_sim.run_simulation(er_exp, new_params, jr_noise_sd=0.0)
        jr_sim.generate_raw(seed=0, noise_fact=noise_fact)
        jr_sim.generate_evoked(er_exp)
        evoked = jr_sim.evoked
        if evoked is None:
            print(f"Warning: Simulation for {param} = {val:.3f} returned no evoked response; skipping.")
            continue
        evoked_list.append(evoked)
        param_vals.append(val)
    return evoked_list, param_vals
    

# ============================================================
# 3. FUNCTION: SAVE SENSITIVITY DATA FOR ONE PARAMETER
# ============================================================
def save_sensitivity_data_for_param(param, default_params, nb_sims, noise_fact, channel_of_interest, jr_param_ranges):
    # Run simulations varying the current parameter.
    evoked_list, param_vals = run_sensitivity_for_param(param, default_params, nb_sims, noise_fact, jr_param_ranges)
    
    if len(evoked_list) == 0:
        print(f"No valid evoked responses for {param}.")
        return None
    
    # Assume all evoked objects share the same time vector.
    times = evoked_list[0].times
    
    # Extract the ERP time series for the chosen channel.
    channel_data = []
    for ev in evoked_list:
        idx = ev.ch_names.index(channel_of_interest)
        channel_data.append(ev.data[idx, :])
    
    # Create an xarray DataArray with dimensions "param_val" and "time".
    da = xr.DataArray(np.array(channel_data),
                     coords={'param_val': param_vals, 'time': times},
                     dims=['param_val', 'time'])
    
    # Compute overall mean ERP (across simulations for this parameter).
    overall_mean_da = da.mean(dim='param_val')
    
    # Calculate plain error (difference from mean)
    diff = da - overall_mean_da
    
    # Calculate gradient of ERP with respect to parameter value
    # This matches the calculation in the image: gradient = np.gradient(data, param_vals, axis=0)
    gradient = np.gradient(da.values, da.param_val, axis=0)
    gradient_da = xr.DataArray(gradient,
                              coords={'param_val': param_vals, 'time': times},
                              dims=['param_val', 'time'])
    
    #try to get gradient on same scale, by normalizing the parameters 

    # Create an xarray Dataset with all the results.
    ds_param = xr.Dataset({
        'ERP': da,
        'error': diff,  # Plain error (difference from mean)
        'gradient': gradient_da  # Gradient of ERP with respect to parameter value
    })
    
    return ds_param






In [2]:
# ============================================================
# 4. LOOP OVER PARAMETERS (ALL) AND SAVE RESULTS
# ============================================================
# Define the parameters you wish to analyze.
params_to_vary = ('A_e', 'A_i', 'b_e', 'b_i', 'a_1', 'a_2', 'a_3', 'a_4', 'C')

# We'll store the sensitivity datasets in a dictionary.
sensitivity_results = {}

for param in params_to_vary:
    ds_param = save_sensitivity_data_for_param(param, default_params, nb_sims, noise_fact, channel_of_interest, jr_param_ranges)
    if ds_param is not None:
        sensitivity_results[param] = ds_param
        # Save each parameter's sensitivity data to a netCDF file.
        filename = f"sensitivity_{nb_sims}_{param}.nc"
        ds_param.to_netcdf(filename)
        print(f"Saved sensitivity data for {param} to {filename}.")

# Next we can merge all results into one xarray Dataset for plotting.
# Note: Merging may require aligning coordinates if the 'param_val' ranges differ.


Running sensitivity analysis for A_e across range [0.0026000000000000003, 0.00975] ...
{'A_e': 0.0026000000000000003, 'A_i': 0.022, 'b_e': 100, 'b_i': 50, 'C': 135, 'a_1': 1.0, 'a_2': 0.8, 'a_3': 0.25, 'a_4': 0.25, 'v_max': 0.05, 'v_0': 0.006}
Reading forward solution from /Users/deepatilwani/Documents/Phd_projects/DCM/Jansen-Rit-Model-Benchmarking-Deep-Learning/notebooks/fsaverage-fwd.fif.gz...
    Reading a source space...
    Computing patch statistics...
    Patch information added...
    Distance information added...
    [done]
    Reading a source space...
    Computing patch statistics...
    Patch information added...
    Distance information added...
    [done]
    2 source spaces read
    Desired named matrix (kind = 3523 (FIFF_MNE_FORWARD_SOLUTION_GRAD)) not available
    Read EEG forward solution (8196 sources, 64 channels, free orientations)
    Source spaces transformed to the forward solution coordinate frame
Reading labels from parcellation...
   read 1 labels from /Use

In [37]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import xarray as xr
import seaborn as sns
import os
from matplotlib.ticker import MaxNLocator, ScalarFormatter
import re

def plot_group_sensitivity(param_group, output_filename, nb_sims, param_ranges=None, conversion_factors=None):
    """
    Plot sensitivity analysis heatmaps for a group of parameters using seaborn's heatmap.
    Parameter values are displayed according to ranges from the standard parameter table.
    
    Parameters:
    -----------
    param_group : tuple or list
        List of parameter names to plot
    output_filename : str
        Filename to save the plot
    nb_sims : int
        Number of simulations (used in file naming)
    param_ranges : dict, optional
        Dictionary with parameter ranges for each parameter
    conversion_factors : dict, optional
        Dictionary with conversion factors to scale parameter values to match the table
    
    Returns:
    --------
    None, saves the plot to output_filename
    """
    # Set font size for consistency
    fontsize = 16  # Reduced font size
    formatter = ScalarFormatter(useOffset=False)
    formatter.set_scientific(True)
    formatter.set_powerlimits((-2, 2))
    
    # Function to format parameter names with subscripts
    def format_param_name(param):
        # Check if the parameter has an underscore
        if '_' in param:
            # Split by underscore and format with subscript
            base, sub = param.split('_', 1)
            return f"{base}$_{{{sub}}}$"
        return param
    
    # Function to format tick labels appropriately based on magnitude and parameter
    def format_tick_label(val, param=None):
        if param == 'C' and val >= 1000:
            # For C parameter with large values, use shorter scientific notation
            return f"{val:.0f}"
        elif abs(val) < 0.01 or abs(val) >= 1000:
            # Scientific notation for very small or very large values
            return f"{val:.2e}"
        else:
            # Fixed point notation for values in between
            return f"{val:.2f}"
    
    # Set up the figure - reduced height per parameter
    n_params = len(param_group)
    fig, axes = plt.subplots(n_params, 3, figsize=(15, 3 * n_params), constrained_layout=True)
    
    # If only one parameter, make axes 2D
    if n_params == 1:
        axes = np.array([axes])
    
    # Store row data for creating consistent y-axes
    row_data = {}
    
    # First pass - collect data for each parameter
    for i, param in enumerate(param_group):
        # Load the dataset for this parameter
        filename = f"sensitivity_{nb_sims}_{param}.nc"
        if not os.path.exists(filename):
            print(f"Warning: File {filename} not found")
            row_data[i] = None
            continue
            
        ds = xr.open_dataset(filename)
        row_data[i] = {
            'param': param,
            'param_vals': sorted(ds['param_val'].values),
            'ds': ds
        }
        
        # Print dataset info for the first parameter
        if i == 0:
            print(f"Dataset variables: {list(ds.data_vars)}")
            print(f"Dataset coordinates: {list(ds.coords)}")
            print(f"Dimensions of ERP: {ds['ERP'].dims}")
    
    # Second pass - create plots with consistent y-axes
    for i, param in enumerate(param_group):
        if row_data[i] is None:
            # Fill row with empty plots
            for j in range(3):
                axes[i, j].set_visible(False)
            continue
        
        ds = row_data[i]['ds']
        param_vals = row_data[i]['param_vals']
        
        # Apply conversion factor if specified
        if conversion_factors and param in conversion_factors:
            display_vals = [val * conversion_factors[param] for val in param_vals]
        else:
            display_vals = param_vals
            
        # Generate ticks for the row
        if param_ranges and param in param_ranges:
            # Use table ranges
            p_min, p_max = param_ranges[param]
            tick_values = np.linspace(p_min, p_max, 6)  # Reduced number of ticks
        else:
            # Use actual values
            tick_values = np.linspace(min(display_vals), max(display_vals), 6)  # Reduced number of ticks
        
        # Format tick labels with special handling for parameter C
        tick_labels = [format_tick_label(val, param) for val in tick_values]
        
        # Extract DataArrays for plotting
        da_erp = ds['ERP']
        da_error = ds['error']
        da_gradient = ds['gradient']
        
        # Create heatmaps for this row
        for j, (da, title_suffix) in enumerate([
            (da_erp, "ERP for"),
            (da_error, "Error for"),
            (da_gradient, "Gradient of ERP for")
        ]):
            ax = axes[i, j]
            
            # Convert to dataframe for seaborn
            df = da.to_dataframe().reset_index()
            df["time"] = df["time"].round(4)
            
            # Pivot data for heatmap (with ascending parameter values)
            hm_data = df.pivot_table(index="param_val", columns="time", values=da.name, aggfunc="mean")
            hm_data = hm_data.sort_index(ascending=True)  # Ensure ascending order
            
            # Create heatmap with adjusted aspect ratio
            hm = sns.heatmap(hm_data, ax=ax, cmap="coolwarm", cbar=True, center=0, 
                           cbar_kws={"shrink": 0.8})  # Smaller colorbar
            
            # Format colorbar
            cbar = hm.collections[0].colorbar
            cbar.ax.tick_params(labelsize=fontsize)
            
            # Format parameter name with subscript for title
            param_formatted = format_param_name(param)
            ax.set_title(f"{title_suffix} {param_formatted}", fontsize=fontsize)
            
            # Set consistent y-ticks across row (in ascending order)
            y_pos = np.linspace(0, len(param_vals), len(tick_values))
            
            # Show y-ticks only for the first column
            if j == 0:
                ax.set_yticks(y_pos)
                ax.set_yticklabels(tick_labels, fontsize=fontsize)
                ax.set_ylabel(f"{param_formatted}", fontsize=fontsize)
            else:
                # Hide y-ticks and labels for columns 2 and 3
                ax.set_yticks([])
                ax.set_yticklabels([])
                ax.set_ylabel("")
            
            # Show x-ticks only for the bottom row
            if i == n_params - 1:
                ax.xaxis.set_major_formatter(formatter)
                ax.xaxis.set_major_locator(MaxNLocator(nbins=6))  # Reduced number of ticks
                ax.tick_params(axis='x', labelsize=fontsize)
                ax.set_xlabel("Time", fontsize=fontsize)
            else:
                # Hide x-ticks and labels for non-bottom rows
                ax.set_xticks([])
                ax.set_xticklabels([])
                ax.set_xlabel("")
        
        # Close the dataset
        ds.close()
    
    # Save the figure
    plt.savefig(output_filename, dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"Sensitivity analysis heatmap saved to {output_filename}")


# Example usage with the provided groups
if __name__ == "__main__":
    # Parameter groups
    group1 = ('A_e', 'A_i', 'b_e', 'b_i')
    group2 = ('a_1', 'a_2', 'a_3', 'a_4', 'C')
    nb_sims = 200
    
    # Define parameter ranges based on the table provided (in standard units)
    param_ranges = {
        'A_e': (2.6, 9.75),    # Excitatory gain in mV
        'A_i': (17.6, 110.0),  # Inhibitory gain in mV
        'b_e': (5, 150),       # Excitatory time constant in s^-1
        'b_i': (25, 75),       # Inhibitory time constant in s^-1
        'C': (65, 1350),       # Connectivity constant
        'a_1': (0.5, 1.5),     # Connectivity parameter
        'a_2': (0.4, 1.2),     # Connectivity parameter
        'a_3': (0.125, 0.375), # Connectivity parameter
        'a_4': (0.125, 0.375)  # Connectivity parameter
    }
    
    # Define conversion factors to scale dataset values to standard units if needed
    # For example, if A_e in dataset is in V but table shows mV, use factor 1000
    conversion_factors = {
        'A_e': 1000,  # Convert V to mV if needed
        'A_i': 1000,  # Convert V to mV if needed
    }
    
    # Plot sensitivity for each group
    plot_group_sensitivity(group1, "sensitivity_group1.png", nb_sims, param_ranges, conversion_factors)
    plot_group_sensitivity(group2, "sensitivity_group2.png", nb_sims, param_ranges, conversion_factors)

Dataset variables: ['ERP', 'error', 'gradient']
Dataset coordinates: ['time', 'param_val']
Dimensions of ERP: ('param_val', 'time')
Sensitivity analysis heatmap saved to sensitivity_group1.png
Dataset variables: ['ERP', 'error', 'gradient']
Dataset coordinates: ['time', 'param_val']
Dimensions of ERP: ('param_val', 'time')
Sensitivity analysis heatmap saved to sensitivity_group2.png
